# ET-600 baseline


This notebook produces the reported **ET-600 baseline** on the frozen split:

- Train: 1–32
- Validation: 33–37
- Test: 38–42

It trains Extra Trees directly from the raw Elliptic feature/class files and writes a summary JSON and a prediction CSV.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

PROJECT_DIR = Path('..')
RAW_DIR = PROJECT_DIR / 'Dataset' / 'raw' / 'elliptic_bitcoin_dataset'

OUT_DIR = PROJECT_DIR / 'outputs' / 'et600_baseline'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_CSV = RAW_DIR / 'elliptic_txs_features.csv'
CLASSES_CSV = RAW_DIR / 'elliptic_txs_classes.csv'

OUT_SUMMARY = OUT_DIR / 'et600_baseline_reproduced_summary.json'
OUT_PRED = OUT_DIR / 'et600_baseline_val_test_predictions.csv'

In [2]:
feat = pd.read_csv(FEATURES_CSV, header=None)
feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(165)]

classes = pd.read_csv(CLASSES_CSV)

df = feat.merge(classes, on='txId', how='left')
df['y'] = pd.to_numeric(df['class'], errors='coerce').map({1: 1, 2: 0})

X = df[[f'f{i}' for i in range(165)]].to_numpy(np.float32)
y = df['y'].to_numpy()
t = df['time_step'].to_numpy(np.int16)

labeled = ~np.isnan(y)

train_mask = labeled & (t >= 1) & (t <= 32)
val_mask = labeled & (t >= 33) & (t <= 37)
test_mask = labeled & (t >= 38) & (t <= 42)

print("train / val / test labeled counts:",
      int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))

train / val / test labeled counts: 28938 4503 6436


In [3]:
model = ExtraTreesClassifier(
    n_estimators=600,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=7,
    n_jobs=-1,
)

model.fit(X[train_mask], y[train_mask].astype(int))

val_p = model.predict_proba(X[val_mask])[:, 1]
test_p = model.predict_proba(X[test_mask])[:, 1]

val_pr = float(average_precision_score(y[val_mask], val_p))
test_pr = float(average_precision_score(y[test_mask], test_p))
test_roc = float(roc_auc_score(y[test_mask], test_p))
test_brier = float(brier_score_loss(y[test_mask], test_p))

summary = {
    'split': {'train': [1, 32], 'val': [33, 37], 'test': [38, 42]},
    'model': {
        'type': 'ExtraTreesClassifier',
        'n_estimators': 600,
        'max_features': 'sqrt',
        'class_weight': 'balanced_subsample',
        'random_state': 7,
    },
    'metrics': {
        'val_pr_auc': val_pr,
        'test_pr_auc': test_pr,
        'test_roc_auc': test_roc,
        'test_brier': test_brier,
    },
}

with open(OUT_SUMMARY, 'w') as f:
    json.dump(summary, f, indent=2)

pred_df = pd.DataFrame({
    'txId': df.loc[val_mask | test_mask, 'txId'].to_numpy(),
    'time_step': df.loc[val_mask | test_mask, 'time_step'].to_numpy(),
    'split': np.where(df.loc[val_mask | test_mask, 'time_step'].to_numpy() <= 37, 'val', 'test'),
    'y': y[val_mask | test_mask].astype(int),
    'et600_p': np.concatenate([val_p, test_p]),
})
pred_df.to_csv(OUT_PRED, index=False)

print(json.dumps(summary['metrics'], indent=2))

{
  "val_pr_auc": 0.9405203366572927,
  "test_pr_auc": 0.8959594088519319,
  "test_roc_auc": 0.9563914828385179,
  "test_brier": 0.02536809439955804
}
